<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

## 05.11 Understanding Inference Control

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)
  
</div>



### 1. Parameter Experiment
So far, we have explored how **model architecture** and **size** affect reasoning and runtime. Now, we turn our attention to the **decoding parameters**, the settings that directly control *how* a model produces text. These parameters control randomness, diversity, repetition, and context usage. Even a small change can make a noticeable difference, especially on resource-constrained devices.

<div class="alert alert-block alert-info">

Use the same model and the same prompt for all runs. Change one parameter at a time to observe its direct effect on output and runtime. Do not forget to fix a seed to make your results reproducible, and use `ollama-gen` for convenience. To do so, provide a value via the `--seed` flag, e.g. `ollama-gen --seed 42`.
</div>

**Example Prompt**: "Write a short poem about the sunrise over the Alps."

**Recommended Model**: `gemma3:1b-it-qat`

| Parameter | Description | Typical Range | What to Observe |
|------------|--------------|----------------|------------------|
| `num_ctx` | Context window size (number of tokens remembered from history) | 512 – 4096 | Does a longer memory improve coherence or slow things down? |
| `num_predict` | Maximum number of tokens to generate (hard stop) | 50 – 512 | How does limiting generation length change response completeness? |
| `temp` | Sampling temperature (controls randomness) | 0.1 – 1.5 | Low = precise and factual, high = creative and varied. |
| `top_k` | Consider only the top-K most likely next tokens | 10 – 100 | Smaller K → conservative and repetitive; larger K → more variation. |
| `top_p` | Nucleus sampling threshold (probability mass) | 0.8 – 1.0 | Controls diversity: higher = more freedom, lower = stricter output. |
| `repeat_penalty` | Penalizes repetition in token choices | 1.0 – 2.0 | Reduces loops and redundant phrases; too high may cut coherence. |

<div class="alert alert-block alert-info">
Once you understand the effect of each parameter, combine them to configure a balanced setup that achieves your preferred trade-off between creativity, precision, and runtime.
</div>

### 2. Model Rambling
Previously, we saw how some models kept generating text long after a natural stopping point. This showed us that not all models have strong internal stopping behavior, and some even need clearer guidance or decoding control to know when to finish.

<div class="alert alert-block alert-info">
Try the following commands to see how much the models ramble when no limits are applied. Here, we do not restrict prediction length:
</div>

```sh
ollama run gemma3:270m " "

ollama run qwen3:1.7b " "

ollama run tinyStories:110m " "
```

**Tip**: Interrupt the generation at any time with `CTRL + C`.

#### 2.1 Why do some models ramble on and on and on and on and ...?

LLMs generate text one token at a time and have no built-in sense of when to stop. Without clear limits, they tend to **over-explain, repeat, or drift off-topic**, especially when the prompt is empty or vague.

*Gemma* is strongly instruction-tuned with conservative default decoding and dependable **stop-token** behavior, so it usually ends cleanly.

*Qwen* is optimized for open-ended, multilingual, and code-heavy continuations. This gives it greater variation but also makes it more prone to long, free-flowing outputs if unconstrained.

*TinyStories* is trained only on continuous narrative text rather than instructions. Since it never learned explicit response boundaries, it tends to keep narrating endlessly when unconstrained, often repeating phrases and extending the story without a clear endpoint.

In practice, we can take measures to limit this:
- **num_predict**: set a hard-stop criterion that terminates inference after a specified number of generated tokens, e.g. `--num_predict 128`
- **Prompt cues**: embed explicit instructions into the prompt, e.g. `--prompt "... End the story with: THE END."`.
- **Temperature**: lower the temperature to tighten the model's output, e.g. `--temp 0.4`
- **Stop sequence**: define a hard stop once a chosen pattern appears in the output, e.g. `--stop '["the end", "THE END"]'`

#### 2.2 Controlling the ramble
Since *TinyStories* does not naturally know when to quit, it often keeps spinning new stories.
The goal is to get a short *TinyStories* generation that ends naturally with `done_reason: "stop"`.
We will still keep `num_predict` as a safety net, but the *actual* stop should be triggered by a **stop sequence**.

We start with a baseline request:

```sh
ollama-gen --model tinyStories:110m --seed 42 --num_predict 512 --prompt "Tell a very short bedtime story about a dragon who"
```

<div class="alert alert-block alert-info">
Run the baseline request and observe its stopping reason.<br>

Now, try to fix the rambling. Add or change the following:

- Adjust the prompt to include a prompt cue<br>
- Reduce the temperature<br>
- Provide a stop sequence that fits your cue and output pattern (e.g. `--stop '["###","THE END","The end.","The end"]'`)
</div>

*Hint*: Notice when the story starts repeating or drifting into a new one. That is your clue for a stopping sequence.

<div class="alert alert-block alert-info">

If you want to challenge yourself, lower `num_predict` and optimize for the smallest value that still produces a clean stop without hitting `done_reason: "length"`.
</div>

#### 2.3 Controlling the ramble: Solution Breakdown
Choosing the right settings is always highly model-specific. With *TinyStories*, we observe a rambling pattern where the model keeps starting new stories with "Once upon a time". This behavior comes from its training data, a set made entirely of children's stories, where that phrase appears frequently. By lowering the temperature, we make the model more predictable. Once it generates *"once"*, it is very likely to continue with the full phrase *"once upon a time"*. Since this marks the beginning of a new story, detecting it is a strong sign that the model has started to ramble. This is why using it as a stop sequence effectively cuts off further unnecessary generations.

----
### Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

**Briefly touch on every row's expected observation. What did you notice?** Answer:

(add your answer here)

**Which parameters had the most significant impact on the output, and which had the least? Why do you think this is the case?** Answer:

(add your answer here)



**If you wanted to make the model more deterministic and faster on the Raspberry Pi, which two parameters would you prioritize adjusting, and why?** Answer:

(add your answer here)

**When increasing both temperature and num_predict, what compound effect did you notice on runtime, output length, and coherence?** Answer:

(add your answer here)

**Which command did you use to finally stop *TinyStories* from rambling? How did you get to it? In case that did not work, briefly describe the steps you took.** Answer:

(add your answer here)

**Optional: Try out some of the models we did not explicitly touch on. How is the user experience?** Answer:

(add your answer here)

**How reliable was your connection to the Raspberry Pi?** Answer:

(add your answer here)

**OPTIONAL: Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)